# Fine-tune GR00T-N1.7 with NVIDIA's `launch_finetune.py`

Vendor path: dataset loading, video decode, normalization, the HF `Trainer` loop,
checkpointing and wandb are all NVIDIA's code. This notebook only supplies the
config -- every hyperparameter and path comes from `configs/finetune_vendor.yaml`
through `training_cfgs.Config` -- stages the dataset, launches the script, and plots the loss.

Companion to `finetune_scratch.ipynb`, which builds the model and training loop by hand.

Run in `$HPCWORK/gr00t-ft/venvs/gr00t` with `module load FFmpeg/7.1.2` loaded, on a GPU.

In [ ]:
!cd /content/finetune-groot/vendor/Isaac-GR00T && git lfs install --local && git lfs pull --include="scripts/deployment/dgpu/wheels/*"
!cd /content/finetune-groot && uv sync

In [21]:
# Google Colab only
!apt-get install -y -qq ffmpeg git-lfs

!git clone https://github.com/svaichu/finetune-groot.git
!cd finetune-groot && mkdir -p vendor && cd vendor && git clone https://github.com/NVIDIA/Isaac-GR00T.git && cd Isaac-GR00T && git checkout n1.7-release && git submodule update --init --recursive 
!cd finetune-groot/vendor/Isaac-GR00T && git lfs install --local && git lfs pull --include="scripts/deployment/dgpu/wheels/*"


!git clone https://github.com/svaichu/robotdataset.git && cd robotdataset && git checkout devel

!cd finetune-groot && uv sync
# !cd finetune-groot && uv pip install -e .

fatal: destination path 'finetune-groot' already exists and is not an empty directory.
fatal: destination path 'Isaac-GR00T' already exists and is not an empty directory.
Updated Git hooks.
Git LFS initialized.
fatal: destination path 'robotdataset' already exists and is not an empty directory.
Resolved 188 packages in 327ms
Uninstalled 1 package in 1ms
Installed 1 package in 2msost1+cu12torch2.7cxx11abifalse (fr
 - flash-attn==2.7.4.post1 (from https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.7cxx11abiFALSE-cp310-cp310-linux_x86_64.whl)
 + flash-attn==2.7.4.post1+cu12torch2.7cxx11abifalse (from https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.7cxx11abiFALSE-cp310-cp310-linux_x86_64.whl)


In [3]:
!pip install training_cfgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 28.5 MB/s eta 0:00:00


In [31]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from huggingface_hub import snapshot_download
from training_cfgs import Config

REPO_ROOT = Path("/content/finetune-groot") if "google.colab" in sys.modules else Path.home() / "finetune-groot"
VENDOR = REPO_ROOT / "vendor" / "Isaac-GR00T"
LAUNCH_SCRIPT = VENDOR / "gr00t" / "experiment" / "launch_finetune.py"

cfg = Config.from_file(REPO_ROOT / "configs" / "finetune_vendor.yaml")
# cfg.apply_args({"training.max_steps": 10000})  # dotted overrides, e.g. for a full run
print(cfg)

Config
  model:
    base_model_path: 'nvidia/GR00T-N1.7-3B' (str)
    embodiment_tag: 'NEW_EMBODIMENT' (str)
    modality_config_path: '/content/finetune-groot/finetune_groot/embodiment_config.py' (str)
    tune_llm: False (bool)
    tune_visual: False (bool)
    tune_projector: True (bool)
    tune_diffusion_model: True (bool)
  data:
    repo_id: 'LSY-lab/stack_without_ft_tact_v4' (str)
    revision: '10f54171c5b08c68bd4af114e4c9a8661ca291d2' (str)
    modality_json: '/content/finetune-groot/configs/modality.json' (str)
  training:
    dataset_path: '/content/gr00t-ft/data/stack_without_ft_tact_v4' (str)
    output_dir: '/hpcwork/yn030245/gr00t-ft/checkpoints' (str)
    experiment_name: 'vendor_notebook_run' (str)
    global_batch_size: 32 (int)
    dataloader_num_workers: 4 (int)
    learning_rate: 0.0001 (float)
    gradient_accumulation_steps: 1 (int)
    max_steps: 10 (int)
    warmup_ratio: 0.05 (float)
    save_steps: 200 (int)
    save_total_limit: 1 (int)
    num_gpus: 1 (int

In [32]:
import gr00t

ModuleNotFoundError: No module named 'gr00t'

## Stage the dataset

The Hub repo is the plain LeRobot v2.1 tree. The vendor script also needs three
files that are not on the Hub: `meta/modality.json` (column slices, copied from
`configs/modality.json`) and `meta/stats.json` + `meta/relative_stats.json`
(generated by the vendor's `stats.py` against the embodiment config). Safe to
re-run: already-downloaded files are skipped and `stats.py` recomputes only stats
whose fingerprint no longer matches. Delete the two stats files to force a rebuild
after changing `embodiment_config.py`.

In [10]:
!pwd

/content


In [19]:
from .finetune_groot.finetune_groot.embodiment_config import MODALITY_CONFIGS

ImportError: attempted relative import with no known parent package

In [12]:
from finetune_groot.finetune_groot import MODALITY_CONFIGS

ModuleNotFoundError: No module named 'finetune_groot'

In [6]:
dataset = Path(cfg.training.dataset_path)

snapshot_download(cfg.data.repo_id, repo_type="dataset", revision=cfg.data.revision, local_dir=dataset)
shutil.copy(cfg.data.modality_json, dataset / "meta" / "modality.json")
subprocess.run([sys.executable, str(VENDOR / "gr00t" / "data" / "stats.py"),
                "--dataset-path", str(dataset),
                "--embodiment-tag", cfg.model.embodiment_tag,
                "--modality-config-path", cfg.model.modality_config_path], check=True)

print(sorted(p.name for p in (dataset / "meta").iterdir()))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 159 files:   0%|          | 0/159 [00:00<?, ?it/s]

CalledProcessError: Command '['/usr/bin/python3', '/content/finetune-groot/vendor/Isaac-GR00T/gr00t/data/stats.py', '--dataset-path', '/content/gr00t-ft/data/stack_without_ft_tact_v4', '--embodiment-tag', 'NEW_EMBODIMENT', '--modality-config-path', '/content/finetune-groot/finetune_groot/embodiment_config.py']' returned non-zero exit status 1.

## Launch

`cfg.flatten()` turns the `model` and `training` groups into the flat flags
`launch_finetune.py` expects (`--tune-projector`, `--no-tune-llm`, `--max-steps 200`, ...).
Output streams below.

In [ ]:
cmd = [sys.executable, str(LAUNCH_SCRIPT), *cfg.flatten(groups=["model", "training"])]
print(" ".join(cmd), "\n")

proc = subprocess.Popen(cmd, cwd=REPO_ROOT, text=True, bufsize=1,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "launch_finetune.py failed"

## Loss

The run is written to `<output_dir>/<experiment_name>/`; the latest
`checkpoint-*/trainer_state.json` holds the full `log_history`.

In [ ]:
run_dir = Path(cfg.training.output_dir) / cfg.training.experiment_name
latest = max(run_dir.glob("checkpoint-*/trainer_state.json"), key=lambda p: int(p.parent.name.split("-")[1]))
log = [r for r in json.loads(latest.read_text())["log_history"] if "loss" in r]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([r["step"] for r in log], [r["loss"] for r in log], marker="o", ms=3)
ax.set_xlabel("step")
ax.set_ylabel("training loss")
ax.set_title(run_dir.name)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()